In [1]:
import pathlib
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.llms import Ollama
from tqdm import tqdm
import warnings
import joblib
import shutil
import os

warnings.filterwarnings('ignore')

c:\Users\My-PC\OneDrive\Desktop\medical-chatbot\env\lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.1.0)/charset_normalizer (3.4.5) doesn't match a supported version!
  warnings.warn(
c:\Users\My-PC\OneDrive\Desktop\medical-chatbot\env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Splitter Function

In [2]:
def split_text(text, chunk_size=500, overlap=100):
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start += chunk_size - overlap
    return chunks


# Convert PDF to Document

In [3]:
import re

documents=[]
path=pathlib.Path('../books')
for file in path.glob("*.pdf"):
    data=PyMuPDFLoader(str(file.absolute()))
    document=data.load()
    for page in document:
        page.page_content = re.sub(r"[^\x20-\x7E]", " ", page.page_content)
        page.page_content = re.sub(r"\s+", " ", page.page_content).encode('utf-8', errors='ignore').decode('utf-8')

        documents.append(page)
    print(f"file name: {file.stem} | total pages: {len(document)}")

print("...finished✅....")

file name: all-medicine-name-list-692 | total pages: 135
file name: Clinical-Pharmacology-11th-Ed-2012-Bennett-Brown | total pages: 622
file name: Davidson's Principles & Practice of Medicine | total pages: 1440
file name: druglist | total pages: 107
file name: Facts for Life | total pages: 216
file name: Gale Encyclopedia of Medicine Vol. 1 (A-B) | total pages: 637
file name: Goodman-Gilmans-The-Pharmacological-Basis-of-Therapeutics-11th-Edition-2006 | total pages: 2047
file name: harrison | total pages: 5113
file name: Indian_Brand_Based_Medicines_for_Medical_Chatbot | total pages: 1
file name: KD Tripathi 8th Edition Essential of Medical Pharmacology | total pages: 1072
MuPDF error: format error: bad data in ahxd: ''

MuPDF error: library error: FT_New_Memory_Face(HiddenHorzOCR): unknown file format

file name: Lippincot Pharmacology 5th edition | total pages: 626
file name: Medical-Terminology-Made-Incredibly-Easy-PDFDrive- | total pages: 434
file name: RANG AND DALE S Pharmacolog

In [4]:
# demo=split_text(documents[0])
# demo[0]

documents[:6]

[Document(metadata={'producer': 'Microsoft: Print To PDF', 'creator': '', 'creationdate': '2022-09-13T10:51:24+05:30', 'source': 'c:\\Users\\My-PC\\OneDrive\\Desktop\\medical-chatbot\\models\\..\\books\\all-medicine-name-list-692.pdf', 'file_path': 'c:\\Users\\My-PC\\OneDrive\\Desktop\\medical-chatbot\\models\\..\\books\\all-medicine-name-list-692.pdf', 'total_pages': 135, 'format': 'PDF 1.7', 'title': 'NLEM, 2022.pdf', 'author': 'Administrator', 'subject': '', 'keywords': '', 'moddate': '2022-09-13T10:51:24+05:30', 'trapped': '', 'modDate': "D:20220913105124+05'30'", 'creationDate': "D:20220913105124+05'30'", 'page': 0}, page_content='National List of Essential Medicines 2022'),
 Document(metadata={'producer': 'Microsoft: Print To PDF', 'creator': '', 'creationdate': '2022-09-13T10:51:24+05:30', 'source': 'c:\\Users\\My-PC\\OneDrive\\Desktop\\medical-chatbot\\models\\..\\books\\all-medicine-name-list-692.pdf', 'file_path': 'c:\\Users\\My-PC\\OneDrive\\Desktop\\medical-chatbot\\models\

In [5]:
documents[6].page_content

'Medicine Level of Healthcare Dosage form(s) and strength(s) 1.4.3 Neostigmine* S,T Tablet 15 mg Injection 0.5 mg/mL 1.4.4 Succinylcholine S,T Injection 50 mg/mL 1.4.5 Vecuronium S,T Powder for injection 4 mg Powder for injection 10 mg Section 2 Analgesics, Antipyretics, Non-steroidal Anti-inflammatory Drugs (NSAIDs), Medicines used to treat Gout and Disease Modifying Agents used in Rheumatoid Disorders 2.1 - Non-opioid Analgesics, Antipyretics and Non-steroidal Anti- inflammatory Drugs Medicine Level of Healthcare Dosage form(s) and strength(s) 2.1.1 Acetylsalicylic acid** P,S,T Tablet 300 mg to 500 mg Effervescent/ Dispersible/ Enteric coated Tablet 300 mg to 500 mg *Neostigmine formulations are also listed in Section 4.2.8 - Antidotes and Other substances used in Management of poisoning/Envenomation - Specific **Acetylsalicylic acid formulations are also listed in - A. Section 5.2.1 - Medicines used in Neurological Disorders Antimigraine medicines B. Section 10.5.1 - Cardiovascular 

# Embedding model (Huggingface)

In [6]:
embeddings=HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    encode_kwargs={"normalize_embeddings": True}
)

joblib.dump(embeddings,"../saved_files/embedding_sentence_transformers.joblib")


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 367.63it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


['../saved_files/embedding_sentence_transformers.joblib']

In [7]:
embeddings=HuggingFaceEmbeddings(
    model_name="BAAI/bge-small-en-v1.5",
    encode_kwargs={"normalize_embeddings": True}
)

joblib.dump(embeddings,"../saved_files/embedding_BAAI.joblib")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 375.52it/s, Materializing param=pooler.dense.weight]                               
BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


['../saved_files/embedding_BAAI.joblib']

In [2]:
embeddings_sentence_transformer = joblib.load("../saved_files/embedding_sentence_transformers.joblib")
embeddings_BAAI = joblib.load("../saved_files/embedding_BAAI.joblib")
print("....embeddings imported✅....")

....embeddings imported✅....


# creating FAISS database

In [9]:
faiss_db_1=None
faiss_db_2=None
for i,doc in enumerate(tqdm(documents)):
    if faiss_db_1 is None and faiss_db_2 is None:
        faiss_db_1 = FAISS.from_documents([doc], embeddings_sentence_transformer)
        faiss_db_2 = FAISS.from_documents([doc], embeddings_BAAI)
    else:
        faiss_db_1.add_documents([doc])
        faiss_db_2.add_documents([doc])

print("...all documents are updated✅...")

faiss_db_path=pathlib.Path('../vector_DB')
faiss_db_1.save_local(faiss_db_path/"medical_pdf_storage_sentence_transformer")
faiss_db_2.save_local(faiss_db_path/"medical_pdf_storage_BAAI")

print(f"database stored locally in {faiss_db_path}")

print("....document count for sentence transformer....")
print("FAISS vectors:", faiss_db_1.index.ntotal)
print("Docstore docs:", len(faiss_db_1.docstore._dict))

print("....document count for BAAI....")
print("FAISS vectors:", faiss_db_2.index.ntotal)
print("Docstore docs:", len(faiss_db_2.docstore._dict))

100%|██████████| 13789/13789 [2:14:20<00:00,  1.71it/s] 


...all documents are updated✅...
database stored locally in ..\vector_DB
....document count for sentence transformer....
FAISS vectors: 13789
Docstore docs: 13789
....document count for BAAI....
FAISS vectors: 13789
Docstore docs: 13789


# Add new doc in FAISS DB

In [3]:
import re

documents=[]
path=pathlib.Path('../books/new_books')
for file in path.glob("*.pdf"):
    data=PyMuPDFLoader(str(file.absolute()))
    document=data.load()
    for page in document:
        page.page_content = re.sub(r"[^\x20-\x7E]", " ", page.page_content)
        page.page_content = re.sub(r"\s+", " ", page.page_content).encode('utf-8', errors='ignore').decode('utf-8')

        documents.append(page)
    print(f"file name: {file.stem} | total pages: {len(document)}")

print("...finished✅....")

file name: 8205Oxford Handbook of Clinical Medicine 10th 2017 Edition_SamanSarKo - Copy | total pages: 911
file name: The_Merck_Manual_of_Diagnosis_and_Therapy_2011_-_19th_Edn....... | total pages: 4114
...finished✅....


In [4]:
faiss_db_1 = FAISS.load_local(
    "../vector_DB/medical_pdf_storage_sentence_transformer",
    embeddings_sentence_transformer,
    allow_dangerous_deserialization=True
    )

faiss_db_2 = FAISS.load_local(
    "../vector_DB/medical_pdf_storage_BAAI",
    embeddings_BAAI,
    allow_dangerous_deserialization=True
    )

for i,doc in enumerate(tqdm(documents)):
        faiss_db_1.add_documents([doc])
        faiss_db_2.add_documents([doc])

print("...all documents are updated✅...")

faiss_db_path=pathlib.Path('../vector_DB')
faiss_db_1.save_local(faiss_db_path/"medical_pdf_storage_sentence_transformer")
faiss_db_2.save_local(faiss_db_path/"medical_pdf_storage_BAAI")

print(f"database stored locally in {faiss_db_path}")

print("....document count for sentence transformer....")
print("FAISS vectors:", faiss_db_1.index.ntotal)
print("Docstore docs:", len(faiss_db_1.docstore._dict))

print("....document count for BAAI....")
print("FAISS vectors:", faiss_db_2.index.ntotal)
print("Docstore docs:", len(faiss_db_2.docstore._dict))

100%|██████████| 5025/5025 [36:59<00:00,  2.26it/s]  


...all documents are updated✅...
database stored locally in ..\vector_DB
....document count for sentence transformer....
FAISS vectors: 18814
Docstore docs: 18814
....document count for BAAI....
FAISS vectors: 18814
Docstore docs: 18814


In [7]:
source_file='../books/new_books'
destination='../books'

for book_path in os.listdir(source_file):
    shutil.move(os.path.join(source_file,book_path),destination)

print(f"books are moved {source_file} to {destination} ✅")

books are moved ../books/new_books to ../books ✅


# RAG pipeline

In [3]:
def rag_pipeline(question):
    # load embeddings 
    embeddings_sentence_transformer = joblib.load("../saved_files/embedding_sentence_transformers.joblib")
    embeddings_BAAI = joblib.load("../saved_files/embedding_BAAI.joblib")

    # load faiss database
    faiss_db_1 = FAISS.load_local(
    "../vector_DB/medical_pdf_storage_sentence_transformer",
    embeddings_sentence_transformer,
    allow_dangerous_deserialization=True
    )

    faiss_db_2 = FAISS.load_local(
    "../vector_DB/medical_pdf_storage_BAAI",
    embeddings_BAAI,
    allow_dangerous_deserialization=True
    )

    
    # perform similarity search with k = 3 sentences
    # the search is based ok cosine similarity
    results_1 = faiss_db_1.similarity_search_with_score(question, k=3)
    results_2 = faiss_db_2.similarity_search_with_score(question, k=3)

    
    # score of documents

        
    result_1_score=max(map(lambda x: x[1],results_1))
    result_2_score=max(map(lambda x: x[1],results_2))

    print("similirity score db1:",result_1_score)
    print("similirity score db2:",result_2_score)

    results = results_1 if result_1_score>=result_2_score else results_2
    
    # convert the vectors to string format
    context ="\n\n".join([i[0].page_content for i in results]) 

    # rag prompt that pass the string as parameter to llama model
    rag_prompt = f"""
    You are a medical assistance chatbot designed to provide general health information in a clear, calm, and responsible manner.


    Your tone should be professional, reassuring, and easy to understand for non-medical users.


    Answer ONLY using the context below.
    Do NOT use outside knowledge.
    If answer is missing, tell that content is not related to medical field.


    context:
    {context}

    Question:
    {question}

    Answer:
    """
    
    # set llama3.2 as LLM model and set temperture as 0 in ollama
    # ollama is platform to run the large language model in local machine
    # llama 3.2 is the lightweight and less capable model. it works on cpu (no GPU required) and it is a 2 billion to 3 billion parameter
    
    llm = Ollama(
        model="llama3.2",
        temperature=0
    )

    response = llm.invoke(rag_prompt)
    return response


In [6]:
query = "i have a severe headache"
print(rag_pipeline(query))

similirity score db1: 0.85362256
similirity score db2: 0.5321678
I'm not able to provide a specific diagnosis or treatment for your severe headache based on the information provided. However, I can tell you that a severe headache can be a symptom of various conditions, including meningitis, subarachnoid hemorrhage, epidural or subdural hematoma, glaucoma, and purulent sinusitis.

In most cases, an abnormal examination should be followed by a computed tomography (CT) or a magnetic resonance imaging (MRI) study to rule out any underlying serious conditions. It's also essential to evaluate your cardiovascular and renal status, eyes, cranial arteries, cervical spine, and psychological state.

If you're experiencing a severe headache, it's crucial to seek medical attention promptly. A healthcare professional can assess your symptoms, perform necessary tests, and provide a proper diagnosis and treatment plan.

In the meantime, if you're feeling calm, try to stay hydrated by drinking plenty o

In [2]:
def rag_pipeline_1(question):
    # load embeddings 
    embeddings=joblib.load("../saved_files/embedding_BAAI.joblib")

    # load faiss database
    faiss_db = FAISS.load_local(
    "../vector_DB/medical_pdf_storage_BAAI",
    embeddings,
    allow_dangerous_deserialization=True
    )

    
    # perform similarity search with k = 3 sentences
    # the search is based ok cosine similarity
    results = faiss_db.similarity_search(question, k=3)

    
    # convert the vectors to string format
    context ="\n\n".join([i.page_content for i in results]) 

    # rag prompt that pass the string as parameter to llama model
    rag_prompt = f"""
    You are a medical assistance chatbot designed to provide general health information in a clear, calm, and responsible manner.


    Your tone should be professional, reassuring, and easy to understand for non-medical users.


    Answer ONLY using the context below.
    Do NOT use outside knowledge.
    If answer is missing, tell that content is not related to medical field.


    context:
    {context}

    Question:
    {question}

    Answer:
    """
    
    # set llama3.2 as LLM model and set temperture as 0 in ollama
    # ollama is platform to run the large language model in local machine
    # llama 3.2 is the lightweight and less capable model. it works on cpu (no GPU required) and it is a 2 billion to 3 billion parameter
    
    llm = Ollama(
        model="llama3.2",
        temperature=0
    )

    response = llm.invoke(rag_prompt)
    return response


In [3]:
query = "i have a headache what the treatment to do?"
print(rag_pipeline_1(query))

I'm not able to provide specific treatment recommendations for your headache. The information I have is general and does not take into account individual circumstances or medical history.

However, I can suggest that you seek medical attention if you experience any of the following:

* Severe or sudden headache
* Headache with stiff neck and fever
* Headache accompanied by vomiting, confusion, or difficulty speaking
* Headache that worsens over time or does not improve with treatment

A healthcare professional can evaluate your symptoms, perform a physical examination, and order diagnostic tests to determine the cause of your headache. They can then provide personalized treatment recommendations based on their diagnosis.

In general, treatment for headaches may include:

* Over-the-counter pain relievers such as acetaminophen or ibuprofen
* Prescription medications such as triptans or ergots for migraines
* Lifestyle changes such as maintaining a consistent sleep schedule, avoiding tri

# Model fine Tuning in Colab

link: https://colab.research.google.com/drive/1YeAS4IG588kv3iZuCGE7cyiTFchpMXf8#scrollTo=qvPmJ_npXZUT

In [2]:
llm = Ollama(
        model="llama3.2",
        temperature=0
    )

In [3]:
def rag_pipeline_ft(question):
    # load embeddings 
    embeddings_sentence_transformer = joblib.load("../saved_files/embedding_sentence_transformers.joblib")
    embeddings_BAAI = joblib.load("../saved_files/embedding_BAAI.joblib")

    # load faiss database
    faiss_db_1 = FAISS.load_local(
    "../vector_DB/medical_pdf_storage_sentence_transformer",
    embeddings_sentence_transformer,
    allow_dangerous_deserialization=True
    )

    faiss_db_2 = FAISS.load_local(
    "../vector_DB/medical_pdf_storage_BAAI",
    embeddings_BAAI,
    allow_dangerous_deserialization=True
    )

    
    # perform similarity search with k = 3 sentences
    # the search is based ok cosine similarity
    results_1 = faiss_db_1.similarity_search_with_score(question, k=3)
    results_2 = faiss_db_2.similarity_search_with_score(question, k=3)

    
    # score of documents

        
    result_1_score=max(map(lambda x: x[1],results_1))
    result_2_score=max(map(lambda x: x[1],results_2))

    print("similirity score db1:",result_1_score)
    print("similirity score db2:",result_2_score)

    results = results_1 if result_1_score>=result_2_score else results_2
    
    # convert the vectors to string format
    context ="\n\n".join([i[0].page_content for i in results]) 

    # rag prompt that pass the string as parameter to llama model
    rag_prompt =  f"""
    You are a medical assistant chatbot designed to provide general health information in a clear, calm, and responsible manner.

    Behavior rules:
    - Show empathy when user mentions illness.
    - Ask follow-up questions if symptoms are unclear.
    - If sufficient symptoms are known, provide general guidance and recommendations.
    - Do NOT diagnose specific diseases.
    - Provide general treatment guidance when relevant medical context is available.
    - Do NOT refuse unnecessarily if question is medical-related.

    Knowledge usage rules:
    - Medical knowledge context is the primary source for treatment guidance.
    - Conversation history contains important symptom information and MUST be used when relevant.
    - If medical knowledge context is empty or insufficient, provide safe general guidance instead of refusing.

    Avoid excessive empathy repetition.
    Keep tone professional, calm, and helpful.
    

    Context:
    {context}

    Question:
    {question}

    Medical Assistant Answer:
    """
    
    # set llama3.2 as LLM model and set temperture as 0 in ollama
    # ollama is platform to run the large language model in local machine
    # llama 3.2 is the lightweight and less capable model. it works on cpu (no GPU required) and it is a 2 billion to 3 billion parameter
    
    

    response = llm.invoke(rag_prompt)
    return response


In [4]:
query = "did you know corona virus"
print(rag_pipeline_ft(query))

similirity score db1: 1.0932046
similirity score db2: 0.6410362
You're referring to Coronaviruses! Yes, I can share some general information about them.

Coronaviruses are a group of viruses that cause respiratory infections, ranging from the common cold to more severe diseases like SARS and MERS. They are called "coronavirus" because they have a crown-like shape under an electron microscope.

Did you know that Coronaviruses are responsible for:

* About 10-20% of all common colds
* Severe Acute Respiratory Syndrome (SARS) and Middle East Respiratory Syndrome (MERS)
* Pneumonia, especially in infants and older adults
* Worsening of chronic bronchitis

However, it's essential to note that Coronaviruses are not the same as COVID-19, which is caused by a specific strain called SARS-CoV-2.

If you have any concerns or symptoms related to a respiratory infection, I recommend speaking with your healthcare provider for personalized advice and guidance.


In [7]:
def rag_pipeline_1(question):
    # load embeddings 
    embeddings=joblib.load("../saved_files/embedding_BAAI.joblib")

    # load faiss database
    faiss_db = FAISS.load_local(
    "../vector_DB/medical_pdf_storage_BAAI",
    embeddings,
    allow_dangerous_deserialization=True
    )

    
    # perform similarity search with k = 3 sentences
    # the search is based ok cosine similarity
    results = faiss_db.similarity_search(question, k=3)

    
    # convert the vectors to string format
    context ="\n\n".join([i.page_content for i in results]) 

    # rag prompt that pass the string as parameter to llama model
    rag_prompt = f"""
    You are a medical assistant.
    Answer only using the given context.
    If the answer is not in the context, say: "The provided content is not related to the medical field."

    Context:
    {context}

    Question:
    {question}

    Answer:
    """
    
    # set llama3.2 as LLM model and set temperture as 0 in ollama
    # ollama is platform to run the large language model in local machine
    # llama 3.2 is the lightweight and less capable model. it works on cpu (no GPU required) and it is a 2 billion to 3 billion parameter
    

    response = llm.invoke(rag_prompt)
    return response


In [10]:
query = "i have a headache what the treatment to do?"
print(rag_pipeline_1(query))

 You are a medical assistant.
    You are a medical assistant.
    Answer only using the given context.
    If the answer is not in the context, say: "The provided content is not related to the medical field."

    Context:
    problem cannot be resolved promptly, necessitating periodic follow- up until further signs appear. CLINICAL EVALUATION OF ACUTE, NEW-ONSET HEADACHE Patients who present with their first severe headache raise entirely different diagnostic possibilities than those with recurrent headaches over many years. In new-onset and severe headaches, the probability of finding a potentially serious cause is considerably greater than in recurrent headache. When a patient complains of an acute, new-onset headache, a number of causes should be considered including meningitis, subarachnoid hemorrhage, epidural or subdural hematoma, glaucoma, and purulent sinusitis. Clinical features of acute, new-onset headache caused by serious underlying conditions are summarized inTable 15-2.

# intent analysis training

In [1]:
from sentence_transformers import SentenceTransformer, util

model = SentenceTransformer('all-MiniLM-L6-v2')



c:\Users\My-PC\OneDrive\Desktop\medical-chatbot\env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\My-PC\OneDrive\Desktop\medical-chatbot\env\lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.1.0)/charset_normalizer (3.4.5) doesn't match a supported version!
  warnings.warn(
c:\Users\My-PC\OneDrive\Desktop\medical-chatbot\env\lib\site-packages\huggingface_hub\file_download.py:130: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\My-PC\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the 

In [3]:
import joblib
joblib.dump(model,"../saved_files/detect_intent.joblib")

['../saved_files/detect_intent.joblib']

In [ ]:
intent_examples = {
    "symptom_description": ["I have fever", "I feel pain"],
    "severity_update": ["it is severe", "getting worse"],
    "treatment_request": ["what medicine", "suggest tablet"],
    "general_query": ["what is fever", "define migraine"]
}

# Precompute
intent_vectors = {
    k: model.encode(v) for k, v in intent_examples.items()
}

In [5]:
def detect_intent(text):
    vec = model.encode(text)

    best_intent = None
    best_score = -1

    for intent, vectors in intent_vectors.items():
        score = max(util.cos_sim(vec, v) for v in vectors)

        if score > best_score:
            best_score = score
            best_intent = intent

    return best_intent

In [16]:
detect_intent('what is paracetmol')

'treatment_request'